# Musanga V2 — Notebook 04 | Illumination-robust U-Net pilot

**Pilot then full four-site LOSO within this SAME notebook:** Train the original ResNet-18 U-Net on three forests and evaluate on the fourth, comparing:

- **M0:** native RGB + historical geometric/mild brightness augmentation
- **M1:** native RGB + realistic stronger illumination/gamma/contrast/smooth-shadow augmentation
- **M2:** M1 + per-site RGB gain referenced to **three training forests**

Sections 00–08 run the small pilot; Sections 09–11 run full four-site LOSO with all eligible crowns, M0/M1 and resumable fold checkpoints.

Default pilot LOSO holdout: **Baego**. This is a fast **single-fold pilot**, not full four-fold LOSO. Individual crown-centred native-resolution patches are used for cheap iteration, and sample sizes are capped. Only annotated crown pixels contribute to loss. Heldout test crowns never select epochs or hyperparameters. Checkpoints are resumable and per-batch/per-epoch progress is printed.

**Important M2 caveat:** Training reference median is fit on the three training forests, but M2 also reads the heldout site's **unlabelled** RGB median to compute its gain; therefore it represents *unsupervised test-image adaptation*, not strict fixed preprocessing. The distinction must be retained in later LOSO reporting.

**Luki:** completely excluded from Notebook 04 fitting and test selection. Its current merged 364-crown reference and forthcoming dark-area additions belong to versioned external evaluation later. No need to touch earlier source imagery or rerun earlier notebook stages to train this pilot.

**Artifacts:** exclusively `MyDrive/Colab Notebooks/Musanga/KVO_work/V2/04_illumination_robust_training/`, with a fingerprinted pilot run directory. No full corrected mosaics or legacy checkpoints are overwritten.

**Execution:** Select a Colab GPU before Section 06. The first run generates and compresses a patch cache once, reusable across all three variants. For an entirely fresh GPU runtime, **run Sections 12–14 only** to train the final four-site M1 model and evaluate Luki. Sections 00–11 are the historical pilot/LOSO experiments and do not need to be rerun. The final model is trained independently from LOSO and is checked against the expanded frozen Luki reference.

## 00 — Setup, pilot budget and source locks

Configure a small **single held-out-forest pilot** first. Default held-out Baego; the other three forests provide spatial training/validation. All outputs are versioned under `KVO_work/V2/04_illumination_robust_training/`. Luki is not loaded or used for fitting.

In [ ]:
%pip -q install rasterio geopandas pyogrio segmentation-models-pytorch scikit-learn tqdm matplotlib
from pathlib import Path
import numpy as np,pandas as pd,geopandas as gpd,rasterio
from rasterio.windows import Window
from rasterio.features import rasterize,geometry_mask
from shapely.geometry import box
from sklearn.metrics import roc_auc_score,average_precision_score,precision_score,recall_score,f1_score,balanced_accuracy_score
from tqdm.auto import tqdm
from IPython.display import display
import segmentation_models_pytorch as smp
import torch,torch.nn as nn
from torch.utils.data import Dataset,DataLoader
import json,hashlib,random,time,math,gc,sys
from google.colab import drive
MY=Path("/content/drive/MyDrive")
if not MY.exists():drive.mount("/content/drive")
ROOT=MY/"Colab Notebooks"/"Musanga"
GEO=ROOT/"data"/"Geo_Ref";AD=GEO/"Annotations"
V2=ROOT/"KVO_work"/"V2"
REF=V2/"01_reference_splits"
HARM=V2/"03_illumination_harmonization"
OUT=V2/"04_illumination_robust_training"
OUT.mkdir(parents=True,exist_ok=True)
CACHE=OUT/"pilot_patch_cache";CACHE.mkdir(exist_ok=True)
SITES={
 "MOS_1_2":("MOS_1_2_kvo.tif","Tree_Annotations_MOS_1_2_kvo.shp"),
 "Yoko":("Yoko_kvo_georef.tif","Tree_Annotations_Yoko_kvo_georef.shp"),
 "Yangambi":("Yangambi_georef.tif","Tree_Annotation_Yangambi_georef.shp"),
 "Baego":("Baego_georef.tif","Tree_Annotation_Baego.shp")}
# PILOT defaults: change only after inspecting first comparisons.
HELDOUT="Baego"
PILOT_TRAIN_CROWNS_PER_SITE=240  # cap across classes, balanced sampling
PILOT_VAL_CROWNS_PER_SITE=90
PILOT_TEST_CROWNS=180
PATCH=256
MAX_EPOCHS=4
PATIENCE=3
BATCH_SIZE=12
SEED=20261013
LR=2e-4
NUM_WORKERS=0  # Colab-safe; on-demand augmentation cheap, avoids multiprocessing state problems
IGNORE=255
MIN_AREA_M2=5.0
RUN_VARIANTS=["M0_native_historical","M1_native_photometric","M2_sitegain_photometric"]
FORCE_REBUILD_CACHE=False
FORCE_RESTART=False
RUN_TRAINING=True
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
assert HELDOUT in SITES
assert (REF/"01_spatial_split_summary.csv").exists(), "Run V2 Notebook 01 first"
assert (HARM/"03_site_luminance_and_fixed_gains.csv").exists(), "Run V2 Notebook 03 Section 02"
if DEVICE=="cpu":
    print("WARNING: CPU detected — select a GPU runtime before training.")
else: print("GPU:",torch.cuda.get_device_name(0))
print("PILOT heldout:",HELDOUT,"epochs:",MAX_EPOCHS,"output:",OUT)
def sha_file(path):
    stat=path.stat()
    return dict(path=str(path),size_bytes=stat.st_size,mtime_ns=stat.st_mtime_ns)
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
seed_all(SEED)


## 01 — Load immutable V2 crown splits

Reconstruct exactly the crowns from V2 Notebook 01 and join the saved spatial split IDs. Invalid geometries stay excluded. This stage does not merge, edit or read any Luki labels; forthcoming dark-area annotations will be handled by a separately versioned external-evaluation update.

In [ ]:
ANN={};SOURCE_FINGERPRINT={}
for site,(filename,shp) in SITES.items():
    rp=GEO/filename;ap=AD/shp
    assert rp.exists() and ap.exists(),site
    with rasterio.open(rp) as src:
        crs=src.crs;bounds=box(*src.bounds)
        assert crs and crs.is_projected
    g=gpd.read_file(ap)
    assert g.crs is not None and "Class_ID" in g.columns
    if g.crs!=crs:g=g.to_crs(crs)
    valid=g.geometry.notna()&(~g.geometry.is_empty)&g.geometry.is_valid
    g=g.loc[valid].copy()
    numerical=pd.to_numeric(g.Class_ID,errors="coerce")
    g=g.loc[numerical.notna()].copy()
    g["class_id"]=numerical.loc[g.index].astype(int)
    g["label"]=(g.class_id==1).astype(np.uint8)
    g["area_m2"]=g.geometry.area
    g=g.loc[(g.area_m2>=MIN_AREA_M2)&g.geometry.intersects(bounds)].copy()
    g["crown_id"]=site+":"+g.index.astype(str)
    reg=pd.read_csv(REF/f"01_{site}_split_register.csv",dtype={"crown_id":"string"})
    assert reg.crown_id.is_unique
    assert set(g.crown_id)==set(reg.crown_id),f"{site} annotation differs from locked Notebook 01 split"
    g=g.merge(reg[["crown_id","split","spatial_block"]],on="crown_id",how="left",validate="one_to_one")
    assert g.split.isin(["training","validation","buffered"]).all()
    ANN[site]=g
    SOURCE_FINGERPRINT[site]=dict(image=sha_file(rp),annotation=sha_file(ap),
                                   split=sha_file(REF/f"01_{site}_split_register.csv"))
    print(site,":",len(g),"crowns; train",sum(g.split=="training"),
          "val",sum(g.split=="validation"),"buffered",sum(g.split=="buffered"))
print("LOSO TEST:",HELDOUT,"uses all annotation classes but only a bounded pilot subsample.")


## 02 — Honest training-only global normalization reference

For M2, calculate the shared reference using only the **three training forests**. Each site's own image brightness index is estimated without species labels. Applying an image-specific gain to the unseen held-out forest is **unsupervised target-image adaptation**, and must be reported as such rather than called strict fixed-preprocessing transfer. M0 and M1 are pure native-RGB transfer.

In [ ]:
stats=pd.read_csv(HARM/"03_site_luminance_and_fixed_gains.csv")
assert set(SITES).issubset(set(stats.site))
training_sites=[s for s in SITES if s!=HELDOUT]
LUMINANCE_MEDIAN={row.site:float(row.site_luminance_median) for _,row in stats.iterrows()}
reference=float(np.median([LUMINANCE_MEDIAN[s] for s in training_sites]))
GLOBAL_GAIN_BOUNDS=(.85,1.15)
GAIN={s:float(np.clip(reference/max(LUMINANCE_MEDIAN[s],1e-6),*GLOBAL_GAIN_BOUNDS)) for s in SITES}
print("Per-fold reference from training sites ONLY:",training_sites,round(reference,4))
print("Per-site gains (test site's own unlabelled RGB statistics imply unsupervised adaptation):",GAIN)
assert HELDOUT not in training_sites


## 03 — Deterministic pilot samples and native-resolution patch cache

Build one shared patch cache per fold from identical source crowns for **all model variants**, with 256 px imagery at each forest's native GSD. Labels are rasterized only from the same training/validation split (all other pixels ignored). Sampling is class-balanced. No full-mosaic masks are constructed, and caches are fingerprinted and reusable.

In [ ]:
def choose_crowns(g,split,n,seed):
    selected=g[g.split==split].copy()
    if n is None or len(selected)<=n:return selected
    rng=np.random.default_rng(seed)
    # Sample both classes; avoid extreme imbalance and keep full indices.
    half=n//2
    p=selected[selected.label==1].index.to_numpy()
    q=selected[selected.label==0].index.to_numpy()
    rng.shuffle(p);rng.shuffle(q)
    p=p[:min(len(p),half)]
    q=q[:min(len(q),n-len(p))]
    if len(p)+len(q)<n:
        remaining=selected.index.difference(np.concatenate([p,q])).to_numpy()
        rng.shuffle(remaining)
        ids=np.concatenate([p,q,remaining[:n-len(p)-len(q)]])
    else:ids=np.concatenate([p,q])
    return selected.loc[ids].copy()

pilot_indices={}
for j,site in enumerate(training_sites):
    g=ANN[site]
    pilot_indices[(site,"training")]=choose_crowns(g,"training",PILOT_TRAIN_CROWNS_PER_SITE,SEED+10*j)
    pilot_indices[(site,"validation")]=choose_crowns(g,"validation",PILOT_VAL_CROWNS_PER_SITE,SEED+100+10*j)
TEST_CROWNS=choose_crowns(ANN[HELDOUT].assign(split="external"),"external",
                          PILOT_TEST_CROWNS,SEED+555)
assert TEST_CROWNS.label.nunique()==2
print("Pilot train",sum(len(pilot_indices[s,"training"]) for s in training_sites),
      "val",sum(len(pilot_indices[s,"validation"]) for s in training_sites),
      "heldout test",len(TEST_CROWNS))
configuration=dict(heldout=HELDOUT,seed=SEED,patch=PATCH,train_cap=PILOT_TRAIN_CROWNS_PER_SITE,
    val_cap=PILOT_VAL_CROWNS_PER_SITE,test_cap=PILOT_TEST_CROWNS,
    source=SOURCE_FINGERPRINT,ref=sha_file(HARM/"03_site_luminance_and_fixed_gains.csv"),
    experiment="v2_pilot_centered_crown_patch_v1")
RUN_ID=hashlib.sha256(json.dumps(configuration,sort_keys=True).encode()).hexdigest()[:12]
RUN_OUT=OUT/f"{HELDOUT}_{RUN_ID}";RUN_OUT.mkdir(exist_ok=True)
NPZ=CACHE/f"pilot_{HELDOUT}_{RUN_ID}.npz"
def centered_window(g,patch,src):
    cx,cy=g.centroid.x,g.centroid.y
    row,col=rasterio.transform.rowcol(src.transform,cx,cy)
    return Window(int(col)-patch//2,int(row)-patch//2,patch,patch)

def generate_cache03():
    xx=[];yy=[];ss=[];sites=[];ids=[]
    # A crop is selected using a crown centroid, but loss includes only the anchor
    # crown's own pixels. This avoids assigning a neighboring split's labels.
    for site in training_sites:
        with rasterio.open(GEO/SITES[site][0]) as src:
            for split_code,split in ((0,"training"),(1,"validation")):
                g=pilot_indices[site,split]
                for _,rec in tqdm(g.iterrows(),total=len(g),desc=f"{site}/{split}",leave=False):
                    w=centered_window(rec.geometry,PATCH,src)
                    rgb=src.read([1,2,3],window=w,boundless=True,fill_value=0)
                    arr=np.moveaxis(rgb,0,-1).astype(np.uint8)
                    original=arr
                    valid=~((original==0).all(axis=2)|
                            (original==255).all(axis=2)|
                            ((original[:,:,0]==0)&(original[:,:,1]==0)&(original[:,:,2]==255)))
                    mask=rasterize([(rec.geometry,int(rec.label))],out_shape=(PATCH,PATCH),
                                   transform=src.window_transform(w),fill=IGNORE,dtype="uint8")
                    mask[~valid]=IGNORE
                    if int((mask!=IGNORE).sum())<12:continue
                    xx.append(arr);yy.append(mask);ss.append(split_code)
                    sites.append(site);ids.append(str(rec.crown_id))
    assert xx and len(set(ss))==2
    X=np.stack(xx);Y=np.stack(yy);S=np.array(ss,np.uint8)
    SI=np.array(sites);CI=np.array(ids)
    print("Caching",len(X),"patches:",round(X.nbytes/1e6,1),"MB raw RGB",flush=True)
    np.savez_compressed(NPZ,X=X,Y=Y,S=S,SI=SI,CI=CI)
    return X,Y,S,SI,CI
if NPZ.exists() and not FORCE_REBUILD_CACHE:
    obj=np.load(NPZ,allow_pickle=False)
    X,Y,S,SI,CI=[obj[k] for k in ["X","Y","S","SI","CI"]]
    print("Reused fingerprinted pilot cache",NPZ)
else:
    X,Y,S,SI,CI=generate_cache03()
assert len(X)==len(Y)==len(S)==len(SI)==len(CI)
assert (S==0).any() and (S==1).any()
print("Patch cache:",len(X),"train:",int((S==0).sum()),"val:",int((S==1).sum()))
pd.DataFrame(dict(site=SI,split=np.where(S==0,"training","validation"),
                  crown_id=CI)).to_csv(RUN_OUT/"04_pilot_patch_registry.csv",index=False)


## 04 — Model, preprocessing and photometric augmentation

Keep the original ResNet-18 U-Net and ImageNet RGB normalization. Compare **M0** historical modest brightness jitter, **M1** randomized brightness/gamma/contrast/channel shifts and smooth shadows, **M2** exactly M1 plus a constant training-site gain. All share architecture, seed, sampled training crowns and optimization settings. Validation never uses random augmentation.

In [ ]:
MEAN=np.array([.485,.456,.406],dtype=np.float32)[None,None,:]
STD=np.array([.229,.224,.225],dtype=np.float32)[None,None,:]
def shadow04(x,rng):
    # Spatially broad and smooth illumination change on the crop.
    # Default local mask variation is differentiable and multiplicative, not recoloring.
    h,w=x.shape[:2]
    yy,xx=np.mgrid[:h,:w].astype(np.float32)
    theta=rng.uniform(0,np.pi)
    normal=(np.cos(theta)*(xx-w/2)+np.sin(theta)*(yy-h/2))
    offset=rng.uniform(-.35,.35)*max(h,w)
    edge_width=rng.uniform(20,90)
    transition=np.clip((normal-offset)/edge_width,-12,12)
    weights=1/(1+np.exp(-transition))
    darkness=rng.uniform(.45,.90)
    return np.clip(x*(darkness+(1-darkness)*weights[:,:,None]),0,1)
class PilotDataset04(Dataset):
    def __init__(self,which,variant,epoch=0):
        self.indices=np.flatnonzero(S==which)
        self.variant=variant;self.epoch=epoch;self.training=(which==0)
    def __len__(self):return len(self.indices)
    def __getitem__(self,j):
        index=int(self.indices[j])
        img=X[index].astype(np.float32)/255.
        mask=Y[index].copy()
        site=str(SI[index])
        if self.variant=="M2_sitegain_photometric":
            img=np.clip(img*GAIN[site],0,1)
        if self.training:
            rng=np.random.default_rng(SEED+self.epoch*1_000_003+index*101)
            if rng.random()<.5:img=img[:,::-1,:].copy();mask=mask[:,::-1].copy()
            if rng.random()<.5:img=img[::-1,:,:].copy();mask=mask[::-1,:].copy()
            k=int(rng.integers(0,4))
            img=np.rot90(img,k).copy();mask=np.rot90(mask,k).copy()
            if self.variant=="M0_native_historical":
                img=np.clip(img*rng.uniform(.90,1.10),0,1)
            else:
                img=np.clip(img*rng.uniform(.70,1.30),0,1)
                img=np.clip((img-.5)*rng.uniform(.80,1.20)+.5,0,1)
                img=np.power(img,float(rng.uniform(.75,1.35)))
                img=np.clip(img*rng.uniform(.92,1.08,size=(1,1,3)),0,1)
                if rng.random()<.55:img=shadow04(img,rng)
        valid=mask!=IGNORE
        if valid.sum()==0:raise RuntimeError("Empty training patch after source QA")
        tensor=np.moveaxis(((img-MEAN)/STD).astype(np.float32),-1,0).copy()
        return torch.from_numpy(tensor),torch.from_numpy(mask.astype(np.int64))
def new_model04():
    return smp.Unet(encoder_name="resnet18",encoder_weights="imagenet",
                    in_channels=3,classes=1).to(DEVICE)
def make_loader04(code,variant,epoch):
    ds=PilotDataset04(code,variant,epoch)
    return DataLoader(ds,batch_size=BATCH_SIZE,shuffle=(code==0),
        pin_memory=DEVICE=="cuda",num_workers=NUM_WORKERS)
print("Each variant has identical architecture, patch inventory and RNG seed.")
print("M2 fold reference learned only from:",training_sites)


## 05 — Fast smoke test (before GPU training)

Validate that each treatment creates finite tensors and that masked targets contain both species classes. Detect configuration errors **before** allocating long GPU runs.

In [ ]:
assert len(set(SI))==len(training_sites)
for variant in RUN_VARIANTS:
    tr=PilotDataset04(0,variant,epoch=1)
    va=PilotDataset04(1,variant,epoch=0)
    a,b=tr[0];c,d=va[0]
    assert a.shape==(3,PATCH,PATCH) and c.shape==(3,PATCH,PATCH)
    assert torch.isfinite(a).all() and torch.isfinite(c).all()
    assert set(torch.unique(b).tolist()).issubset({0,1,IGNORE})
    print(variant,"train batches",math.ceil(len(tr)/BATCH_SIZE),
          "val batches",math.ceil(len(va)/BATCH_SIZE),
          "x range",round(float(a.min()),2),round(float(a.max()),2))
print("Smoke test passed.")


## 06 — Resumable pilot training with per-epoch progress

Run M0/M1/M2 sequentially with an identical pilot budget. Checkpoints are saved **every epoch**, including optimizer, scheduler-free training state, AMP scaler, early-stopping count and best validation AP; reruns resume instead of starting over. Console output is flushed each epoch. No held-out-forest labels are used for checkpoint selection.

In [ ]:
def score_valid04(model,loader):
    model.eval();true=[];pred=[]
    with torch.inference_mode():
        for x,y in loader:
            x=x.to(DEVICE,non_blocking=True)
            with torch.amp.autocast("cuda",enabled=DEVICE=="cuda"):
                out=torch.sigmoid(model(x).squeeze(1))
            mask=(y!=IGNORE)
            true.append(y[mask].numpy().astype(np.uint8))
            pred.append(out.detach().cpu()[mask].numpy().astype(np.float32))
    target=np.concatenate(true);prob=np.concatenate(pred)
    return dict(ap=float(average_precision_score(target,prob)) if target.sum() else np.nan,
        auc=float(roc_auc_score(target,prob)) if len(np.unique(target))==2 else np.nan,
        pixels=int(len(target)))
def train_variant04(variant):
    folder=RUN_OUT/variant;folder.mkdir(parents=True,exist_ok=True)
    progress=folder/"last_training_state.pt";bestfile=folder/"best_model.pt"
    history_file=folder/"epoch_history.csv";marker=folder/"pilot_complete.json"
    if marker.exists() and bestfile.exists() and not FORCE_RESTART:
        print("REUSING COMPLETED PILOT:",variant,flush=True)
        return json.loads(marker.read_text())
    seed_all(SEED)
    model=new_model04()
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
    scaler=torch.amp.GradScaler("cuda",enabled=DEVICE=="cuda")
    history=[];start=1;best=-1.;wait=0
    if progress.exists() and not FORCE_RESTART:
        state=torch.load(progress,map_location=DEVICE,weights_only=False)
        assert state["variant"]==variant and state["run_id"]==RUN_ID
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])
        best=float(state["best"]);wait=int(state["wait"])
        start=int(state["epoch"])+1
        history=state["history"]
        print("RESUMING:",variant,"epoch",start,"best AP",round(best,4),flush=True)
    for epoch in range(start,MAX_EPOCHS+1):
        begin=time.monotonic();model.train();losses=[]
        loader=make_loader04(0,variant,epoch)
        for bi,(x,y) in enumerate(loader,1):
            x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda",enabled=DEVICE=="cuda"):
                logits=model(x).squeeze(1)
                valid=(y!=IGNORE)
                loss=nn.functional.binary_cross_entropy_with_logits(logits[valid],y[valid].float())
            scaler.scale(loss).backward()
            scaler.step(optimizer);scaler.update()
            losses.append(float(loss.item()))
            if bi==1 or bi%25==0 or bi==len(loader):
                print(f"  {variant} epoch {epoch} batch {bi}/{len(loader)} loss {np.mean(losses):.4f}",
                      flush=True)
        validation=score_valid04(model,make_loader04(1,variant,0))
        improved=np.isfinite(validation["ap"]) and validation["ap"]>best
        if improved:
            best=validation["ap"];wait=0
            torch.save(dict(model=model.state_dict(),epoch=epoch,variant=variant,
                            run_id=RUN_ID,best_validation_ap=best),bestfile)
        else:wait+=1
        info=dict(epoch=epoch,train_loss=float(np.mean(losses)),validation_ap=validation["ap"],
                  validation_auc=validation["auc"],validation_pixels=validation["pixels"],
                  seconds=float(time.monotonic()-begin),best_validation_ap=best,wait=wait)
        history.append(info)
        pd.DataFrame(history).to_csv(history_file,index=False)
        torch.save(dict(variant=variant,run_id=RUN_ID,epoch=epoch,
            model=model.state_dict(),optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),
            best=best,wait=wait,history=history),progress)
        print(f"PROGRESS {variant} epoch {epoch}/{MAX_EPOCHS}: loss={info['train_loss']:.4f}, "
              f"AP={info['validation_ap']:.4f}, AUC={info['validation_auc']:.4f}, "
              f"best_AP={best:.4f}, {info['seconds']:.1f}s",flush=True)
        if wait>=PATIENCE:
            print("EARLY STOP:",variant,"after",epoch,"epochs",flush=True);break
    if not bestfile.exists():raise RuntimeError(f"Best checkpoint never saved for {variant}")
    summary=dict(variant=variant,run_id=RUN_ID,heldout=HELDOUT,
                 epochs_completed=len(history),best_validation_ap=best,
                 selection_basis="spatial validation pixels on three training forests",
                 heldout_labels_used_for_selection=False)
    marker.write_text(json.dumps(summary,indent=2))
    del model,optimizer;gc.collect()
    if DEVICE=="cuda":torch.cuda.empty_cache()
    return summary
TRAIN_RESULTS04=[]
if RUN_TRAINING:
    assert DEVICE=="cuda","Change runtime to GPU; refusing long CPU training."
    for variant in RUN_VARIANTS:
        print("\nSTART VARIANT:",variant,flush=True)
        TRAIN_RESULTS04.append(train_variant04(variant))
    display(pd.DataFrame(TRAIN_RESULTS04))
else:print("RUN_TRAINING=False. Smoke test only.")


## 07 — Small independent Baego test; no selection on test

Evaluate each trained variant on an identical, class-balanced sample from the **held-out fourth forest**. Each crown is scored using mean predicted probability within its original labelled polygon on a native-resolution patch. Results are diagnostic (pilot sample), **not full wall-to-wall canopy accuracy or complete LOSO**. Do not choose hyperparameters based on Luki.

In [ ]:
def test_crowns04(model,variant,test_g):
    model.eval();rows=[]
    with rasterio.open(GEO/SITES[HELDOUT][0]) as src:
        for _,rec in tqdm(test_g.iterrows(),total=len(test_g),desc=f"{variant} heldout crowns"):
            w=centered_window(rec.geometry,PATCH,src)
            raw=np.moveaxis(src.read([1,2,3],window=w,boundless=True,fill_value=0),0,-1)
            valid=~((raw==0).all(axis=2)|(raw==255).all(axis=2)|
                ((raw[:,:,0]==0)&(raw[:,:,1]==0)&(raw[:,:,2]==255)))
            interior=geometry_mask([rec.geometry],out_shape=(PATCH,PATCH),
                                    transform=src.window_transform(w),invert=True)&valid
            if interior.sum()<12:continue
            img=raw.astype(np.float32)/255.
            if variant=="M2_sitegain_photometric":
                img=np.clip(img*GAIN[HELDOUT],0,1)
            normalized=np.moveaxis((img-MEAN)/STD,-1,0).astype(np.float32).copy()
            with torch.inference_mode(),torch.amp.autocast("cuda",enabled=DEVICE=="cuda"):
                p=torch.sigmoid(model(torch.from_numpy(normalized[None]).to(DEVICE))).squeeze().float().cpu().numpy()
            score=float(np.mean(p[interior]))
            rows.append(dict(site=HELDOUT,crown_id=str(rec.crown_id),label=int(rec.label),
                predicted_probability=score,valid_pixels=int(interior.sum()),
                variant=variant,experiment="balanced_heldout_crown_pilot"))
    return pd.DataFrame(rows)
TEST_PRED04=[]
if RUN_TRAINING:
    for variant in RUN_VARIANTS:
        folder=RUN_OUT/variant
        bestfile=folder/"best_model.pt"
        assert bestfile.exists(),f"Missing {bestfile}"
        seed_all(SEED);model=new_model04()
        ckpt=torch.load(bestfile,map_location=DEVICE,weights_only=False)
        assert ckpt["run_id"]==RUN_ID and ckpt["variant"]==variant
        model.load_state_dict(ckpt["model"])
        df=test_crowns04(model,variant,TEST_CROWNS)
        assert df.label.nunique()==2
        df.to_csv(folder/"heldout_pilot_crown_predictions.csv",index=False)
        TEST_PRED04.append(df)
        print("EVALUATED",variant,"crowns",len(df),flush=True)
        del model;gc.collect()
        if DEVICE=="cuda":torch.cuda.empty_cache()
    HOLDOUT04=pd.concat(TEST_PRED04,ignore_index=True)
    HOLDOUT04.to_csv(RUN_OUT/"04_heldout_pilot_crown_predictions.csv",index=False)


## 08 — Compare pilot results and save handoff

Compare validation-selected checkpoints on a held-out forest, alongside run-time budgets. Keep results labelled as **pilot**, and make no claims about Luki or wall-to-wall canopy cover. Subsequent complete four-fold LOSO remains the authority; revised Luki dark-area annotations must have explicit reference versions.

In [ ]:
def summarize04(d):
    y=d.label.to_numpy(dtype=int)
    p=d.predicted_probability.to_numpy(float)
    q=(p>=.5).astype(int)
    return dict(n=len(y),musanga=int(y.sum()),other=int((1-y).sum()),
        auc=float(roc_auc_score(y,p)),average_precision=float(average_precision_score(y,p)),
        precision=float(precision_score(y,q,zero_division=0)),
        recall=float(recall_score(y,q,zero_division=0)),
        balanced_accuracy=float(balanced_accuracy_score(y,q)),
        f1=float(f1_score(y,q,zero_division=0)),
        false_positives=int(((y==0)&(q==1)).sum()),
        false_negatives=int(((y==1)&(q==0)).sum()))
if RUN_TRAINING:
    SUMMARY04=pd.DataFrame([dict(variant=v,**summarize04(HOLDOUT04[HOLDOUT04.variant==v]))
                           for v in RUN_VARIANTS])
    SUMMARY04.to_csv(RUN_OUT/"04_heldout_pilot_comparison.csv",index=False)
    display(SUMMARY04.round(3))
manifest=dict(stage="Musanga V2 Notebook 04: pilot U-Net",
    heldout=HELDOUT,training_sites=training_sites,
    architecture="segmentation_models_pytorch.Unet resnet18 imagenet",
    spatial_splits_source=str(REF),source_files=SOURCE_FINGERPRINT,
    run_id=RUN_ID,run_dir=str(RUN_OUT),
    patch_size=PATCH,seed=SEED,
    training_caps_per_site=PILOT_TRAIN_CROWNS_PER_SITE,
    validation_caps_per_site=PILOT_VAL_CROWNS_PER_SITE,
    heldout_crown_pilot_cap=PILOT_TEST_CROWNS,
    training_epochs=MAX_EPOCHS,patience=PATIENCE,variants=RUN_VARIANTS,
    normalization_reference_sites=training_sites,
    normalization_reference_luminance=reference,site_gains=GAIN,
    m2_interpretation="unlabeled held-out image statistics used for site gain: unsupervised target adaptation",
    heldout_selection_policy="test labels never choose checkpoint or photometric settings",
    luki_policy="Luki excluded from model fitting, checkpoint selection and pilot testing; new dark annotations are future versioned external references",
    spatial_qa="single-crown labelled patches, all other pixels ignored, split buffers inherited from Notebook 01",
    output_policy="All generated files exclusively under Musanga/KVO_work/V2")
(RUN_OUT/"04_pilot_manifest.json").write_text(json.dumps(manifest,indent=2))
print("Pilot run:",RUN_ID)
print("Outputs:",RUN_OUT)
print("NEXT: inspect pilot results and decide whether full four-fold LOSO is warranted.")


## 09 — Full four-site LOSO: configuration and safe state isolation

This is **the full experiment inside Notebook 04**, not a new notebook. It uses all eligible Notebook 01 **training** and **validation** crowns in each of the four LOSO folds, and tests **all eligible held-out crowns**, with no artificial class balancing. Two variants: M0 original modest augmentation, M1 stronger photometric augmentation. Pixel-level validation AP selects checkpoints; independent held-out **crown-level ROC-AUC and AP** are primary ranking metrics. Additional F1 at 0.5 is descriptive only.

**Reproducibility and efficiency:** reuse Section 03/04 cache and U-Net functions; write into a separate `full_LOSO/` folder; freeze pilot globals after this stage; checkpoints every epoch and short streamed training progress. Default maximum 20 epochs and early stopping after four non-improving epochs. Training-crown **centre crops** approximate sampled crown recognition, not wall-to-wall U-Net pixel evaluation. Luki remains untouched.

In [ ]:
# Run the existing pilot once, then run Sections 09–11 to execute full LOSO.
RUN_FULL_LOSO=True
FULL_FOLDS=["Baego","MOS_1_2","Yoko","Yangambi"]
FULL_VARIANTS=["M0_native_historical","M1_native_photometric"]
FULL_MAX_EPOCHS=20
FULL_PATIENCE=4
FULL_BATCH_SIZE=12
FULL_ROOT=OUT/"full_LOSO"
FULL_ROOT.mkdir(parents=True,exist_ok=True)
assert len(FULL_FOLDS)==4 and set(FULL_FOLDS)==set(SITES)
assert set(FULL_VARIANTS).issubset(set(RUN_VARIANTS))
FULL_SUMMARY=FULL_ROOT/"04_full_LOSO_fold_metrics.csv"
# Sections 03–08 deliberately used shared notebook globals for simple pilot reuse.
# Snapshot these before substituting a complete fold and restore in Section 11.
PILOT_GLOBALS04={k:globals()[k] for k in [
    "HELDOUT","training_sites","reference","GAIN","RUN_ID","RUN_OUT",
    "NPZ","pilot_indices","TEST_CROWNS","X","Y","S","SI","CI",
    "RUN_VARIANTS","MAX_EPOCHS","PATIENCE","BATCH_SIZE","FORCE_RESTART"]}
print("Full LOSO folds:",FULL_FOLDS, "| models:",FULL_VARIANTS,flush=True)
print("Max epochs:",FULL_MAX_EPOCHS,"early-stop patience:",FULL_PATIENCE,flush=True)
print("Outputs:",FULL_ROOT,flush=True)

## 10 — Execute full LOSO with independent cache and checkpoint per fold

**Run once.** Each fold computes the training-only normalization reference, builds or reuses its own fingerprinted full-crown patch cache, trains or resumes M0 and M1, and evaluates the entire held-out forest using the existing immutable crown annotations. Completion is streamed after every fold. The pilot is never overwritten.

**Interpretation:** M0/M1 use native RGB for held-out testing (no target-domain normalization); the fold reference is recorded for reproducibility but unused by M0/M1. No SAM and no Luki in this workflow. The held-out labels are accessed **only after training** for evaluation, not to select the best epoch.

In [ ]:
def fold_cache04():
    # All eligible crowns, including natural species prevalences; split buffers excluded.
    global X,Y,S,SI,CI,NPZ
    if NPZ.exists() and not FORCE_REBUILD_CACHE:
        with np.load(NPZ,allow_pickle=False) as obj:
            X,Y,S,SI,CI=[obj[k] for k in ["X","Y","S","SI","CI"]]
        print("CACHED full-crown patches",len(X),NPZ.name,flush=True)
    else:
        X,Y,S,SI,CI=generate_cache03()
    assert set(SI)==set(training_sites)
    assert (S==0).any() and (S==1).any()
    return len(X)

def evaluate_fold04(variant):
    ck=RUN_OUT/variant/"best_model.pt"
    assert ck.exists(),f"Missing selected checkpoint: {ck}"
    seed_all(SEED)
    model=new_model04()
    state=torch.load(ck,map_location=DEVICE,weights_only=False)
    assert state["run_id"]==RUN_ID and state["variant"]==variant
    model.load_state_dict(state["model"])
    pred=test_crowns04(model,variant,TEST_CROWNS)
    del model
    if DEVICE=="cuda":torch.cuda.empty_cache()
    gc.collect()
    assert len(pred)>0 and pred.label.nunique()==2
    assert pred.crown_id.is_unique
    assert set(pred.crown_id).issubset(set(TEST_CROWNS.crown_id))
    pred.to_csv(RUN_OUT/variant/"full_heldout_crown_predictions.csv",index=False)
    return dict(fold=HELDOUT,variant=variant,
                reference_crowns=int(len(TEST_CROWNS)),
                evaluated_crowns=int(len(pred)),
                **summarize04(pred))

def run_full_fold04(holdout):
    global HELDOUT,training_sites,reference,GAIN,RUN_ID,RUN_OUT,NPZ
    global pilot_indices,TEST_CROWNS,RUN_VARIANTS,MAX_EPOCHS,PATIENCE,BATCH_SIZE
    assert holdout in SITES
    HELDOUT=holdout
    training_sites=[s for s in SITES if s!=HELDOUT]
    assert len(training_sites)==3
    # This reference uses ONLY the three development forests of this fold.
    reference=float(np.median([LUMINANCE_MEDIAN[s] for s in training_sites]))
    GAIN={s:float(np.clip(reference/max(LUMINANCE_MEDIAN[s],1e-6),
                           *GLOBAL_GAIN_BOUNDS)) for s in SITES}
    pilot_indices={(s,sp):ANN[s][ANN[s].split==sp].copy()
                   for s in training_sites for sp in ("training","validation")}
    TEST_CROWNS=ANN[HELDOUT].copy()  # ALL annotated crowns; original prevalence
    RUN_VARIANTS=FULL_VARIANTS.copy()
    MAX_EPOCHS=FULL_MAX_EPOCHS
    PATIENCE=FULL_PATIENCE
    BATCH_SIZE=FULL_BATCH_SIZE
    configuration=dict(stage="V2_04_full_loso_crown_patches_v1",
      fold=HELDOUT,train_sites=training_sites,
      train_ids={s:sorted(pilot_indices[s,"training"].crown_id.astype(str).tolist())
                 for s in training_sites},
      validation_ids={s:sorted(pilot_indices[s,"validation"].crown_id.astype(str).tolist())
                      for s in training_sites},
      patch=PATCH,seed=SEED,
      source={s:SOURCE_FINGERPRINT[s] for s in training_sites},
      photo_aug_version=1)
    RUN_ID=hashlib.sha256(json.dumps(configuration,sort_keys=True).encode()).hexdigest()[:12]
    RUN_OUT=FULL_ROOT/f"{HELDOUT}_{RUN_ID}"
    RUN_OUT.mkdir(exist_ok=True)
    NPZ=CACHE/f"full_{HELDOUT}_{RUN_ID}.npz"
    # A complete fold is resumable independently, including if Colab disconnects.
    summary_path=RUN_OUT/"04_fold_results.csv"
    if summary_path.exists():
        existing=pd.read_csv(summary_path)
        if set(existing.variant)==set(FULL_VARIANTS) and len(existing)==len(FULL_VARIANTS):
            print("REUSING COMPLETE FULL LOSO FOLD",HELDOUT,flush=True)
            return existing
    n=fold_cache04()
    print(f"\nFULL FOLD {HELDOUT}: train={int((S==0).sum())}, "
          f"val={int((S==1).sum())}, test={len(TEST_CROWNS)}; cached {n}",flush=True)
    results=[]
    for variant in FULL_VARIANTS:
        print("TRAIN:",HELDOUT,variant,flush=True)
        selected=train_variant04(variant)
        print("TEST:",HELDOUT,variant,flush=True)
        scores=evaluate_fold04(variant)
        scores["best_training_validation_ap"]=selected["best_validation_ap"]
        scores["run_id"]=RUN_ID
        results.append(scores)
        # Write cumulative results after every model, even if later folds fail.
        pd.DataFrame(results).to_csv(summary_path,index=False)
        print("DONE",HELDOUT,variant,
              "crown AUC",round(scores["auc"],4),
              "AP",round(scores["average_precision"],4),
              "F1@0.5",round(scores["f1"],4),flush=True)
    return pd.DataFrame(results)

ALL_FULL04=[]
if RUN_FULL_LOSO:
    assert DEVICE=="cuda","Choose a GPU runtime for full LOSO."
    for fold in FULL_FOLDS:
        print("\n"+"="*55+"\nFULL LOSO HELD-OUT:",fold,flush=True)
        df=run_full_fold04(fold)
        ALL_FULL04.append(df)
        pd.concat(ALL_FULL04,ignore_index=True).to_csv(FULL_SUMMARY,index=False)
    print("\nAll four folds finished. Saved:",FULL_SUMMARY,flush=True)
else:
    print("Full LOSO disabled. Set RUN_FULL_LOSO=True and rerun section 10.")

## 11 — Aggregate ROC-AUC / AP, calibration-sensitive metrics and versioned completion manifest

Report crown-level ROC-AUC and AP as **primary**, alongside threshold-dependent precision, recall, F1 and balanced accuracy. Macro averaging across forests avoids treating the largest forest as automatically most important. Flag missing predictions; preserve per-crown outputs for later absolute RGB brightness joins. **No claim of external Luki accuracy here.**

In [ ]:
if FULL_SUMMARY.exists():
    result=pd.read_csv(FULL_SUMMARY)
    assert set(result.fold)==set(FULL_FOLDS),(
        "Full LOSO incomplete; existing folds:",sorted(result.fold.unique()))
    assert len(result)==len(FULL_FOLDS)*len(FULL_VARIANTS)
    order=["fold","variant","reference_crowns","evaluated_crowns",
           "auc","average_precision","precision","recall","balanced_accuracy","f1",
           "false_positives","false_negatives","best_training_validation_ap"]
    display(result[order].round(4))
    macro=result.groupby("variant",as_index=False).agg(
        folds=("fold","nunique"),
        macro_crown_auc=("auc","mean"),
        macro_crown_ap=("average_precision","mean"),
        macro_f1_at_0p5=("f1","mean"),
        macro_balanced_accuracy_at_0p5=("balanced_accuracy","mean"),
        total_crowns=("evaluated_crowns","sum"),
        missing_crowns=("reference_crowns",lambda x:0)) # recomputed below
    missing=result.groupby("variant").apply(
        lambda d:int((d.reference_crowns-d.evaluated_crowns).sum()),
        include_groups=False)
    macro["missing_crowns"]=macro.variant.map(missing)
    macro.to_csv(FULL_ROOT/"04_full_LOSO_macro_summary.csv",index=False)
    display(macro.round(4))
    contract=dict(stage="Musanga V2/04 complete LOSO using manual crowns",
      folds=FULL_FOLDS,variants=FULL_VARIANTS,heldout_Luki=False,
      patch_size_px=PATCH,
      trained_on="All eligible train split crowns in three forests per fold",
      validated_on="All eligible validation split crowns in three forests per fold",
      test_on="All eligible reference crowns from the fourth forest",
      reference_split_source=str(REF),
      primary_metrics=["heldout_crown_ROC_AUC","heldout_crown_average_precision"],
      secondary_metrics=["F1_at_probability_0.5","balanced_accuracy","precision","recall"],
      image_input="native UAV RGB only; no SAM, no spatial resampling",
      threshold_policy="fixed 0.5 is diagnostic, not chosen on held-out test",
      production_warning="Crown-centred patch pilot/training is NOT full orthomosaic pixel-level test",
      luki="held aside for future, versioned new dark-region annotations",
      full_metrics=str(FULL_SUMMARY),
      macro_metrics=str(FULL_ROOT/"04_full_LOSO_macro_summary.csv"))
    (FULL_ROOT/"04_full_LOSO_manifest.json").write_text(json.dumps(contract,indent=2))
    print("Full LOSO finished; all pilot outputs remain intact.",flush=True)
else:
    print("Section 10 has not yet produced full LOSO metrics.")
# Restore pilot state, so revisiting earlier sections does not cross-contaminate.
if "PILOT_GLOBALS04" in globals():
    globals().update(PILOT_GLOBALS04)
    print("Restored pilot configuration and arrays.")

## 12 — Fresh-runtime setup for the final four-site M1 model

**Start here in a NEW Colab GPU runtime.** This section independently mounts Drive, loads the immutable four-site Notebook 01 references and split registers, defines the original U-Net training and patch functions, and locks the versioned Luki reference. It does **not** execute the pilot or four LOSO folds. After this cell, run Sections **13 and 14 only**.

The final training cache is separately fingerprinted and resumable; completed pilot and LOSO checkpoints remain untouched. No GPU is needed until Section 13, but select a GPU runtime before starting.

In [ ]:
# FRESH-RUNTIME BOOTSTRAP: only the essential earlier DEFINITIONS are used.
# No pilot inference or four-fold LOSO is launched by this section.
%pip -q install rasterio geopandas pyogrio segmentation-models-pytorch scikit-learn tqdm matplotlib
from pathlib import Path
import numpy as np,pandas as pd,geopandas as gpd,rasterio
from rasterio.windows import Window
from rasterio.features import rasterize,geometry_mask
from shapely.geometry import box
from sklearn.metrics import roc_auc_score,average_precision_score,precision_score,recall_score,f1_score,balanced_accuracy_score
from tqdm.auto import tqdm
from IPython.display import display
import segmentation_models_pytorch as smp
import torch,torch.nn as nn
from torch.utils.data import Dataset,DataLoader
import json,hashlib,random,time,math,gc,sys
from google.colab import drive
MY=Path("/content/drive/MyDrive")
if not MY.exists():drive.mount("/content/drive")
ROOT=MY/"Colab Notebooks"/"Musanga"
GEO=ROOT/"data"/"Geo_Ref";AD=GEO/"Annotations"
V2=ROOT/"KVO_work"/"V2"
REF=V2/"01_reference_splits"
HARM=V2/"03_illumination_harmonization"
OUT=V2/"04_illumination_robust_training"
OUT.mkdir(parents=True,exist_ok=True)
CACHE=OUT/"pilot_patch_cache";CACHE.mkdir(exist_ok=True)
SITES={
 "MOS_1_2":("MOS_1_2_kvo.tif","Tree_Annotations_MOS_1_2_kvo.shp"),
 "Yoko":("Yoko_kvo_georef.tif","Tree_Annotations_Yoko_kvo_georef.shp"),
 "Yangambi":("Yangambi_georef.tif","Tree_Annotation_Yangambi_georef.shp"),
 "Baego":("Baego_georef.tif","Tree_Annotation_Baego.shp")}
# PILOT defaults: change only after inspecting first comparisons.
HELDOUT="Baego"
PILOT_TRAIN_CROWNS_PER_SITE=240  # cap across classes, balanced sampling
PILOT_VAL_CROWNS_PER_SITE=90
PILOT_TEST_CROWNS=180
PATCH=256
MAX_EPOCHS=4
PATIENCE=3
BATCH_SIZE=12
SEED=20261013
LR=2e-4
NUM_WORKERS=0  # Colab-safe; on-demand augmentation cheap, avoids multiprocessing state problems
IGNORE=255
MIN_AREA_M2=5.0
RUN_VARIANTS=["M0_native_historical","M1_native_photometric","M2_sitegain_photometric"]
FORCE_REBUILD_CACHE=False
FORCE_RESTART=False
RUN_TRAINING=False
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
assert HELDOUT in SITES
assert (REF/"01_spatial_split_summary.csv").exists(), "Run V2 Notebook 01 first"
assert (HARM/"03_site_luminance_and_fixed_gains.csv").exists(), "Run V2 Notebook 03 Section 02"
if DEVICE=="cpu":
    print("WARNING: CPU detected — select a GPU runtime before training.")
else: print("GPU:",torch.cuda.get_device_name(0))
print("Fresh-runtime final-M1 bootstrap; output:",OUT)
def sha_file(path):
    stat=path.stat()
    return dict(path=str(path),size_bytes=stat.st_size,mtime_ns=stat.st_mtime_ns)
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
seed_all(SEED)

ANN={};SOURCE_FINGERPRINT={}
for site,(filename,shp) in SITES.items():
    rp=GEO/filename;ap=AD/shp
    assert rp.exists() and ap.exists(),site
    with rasterio.open(rp) as src:
        crs=src.crs;bounds=box(*src.bounds)
        assert crs and crs.is_projected
    g=gpd.read_file(ap)
    assert g.crs is not None and "Class_ID" in g.columns
    if g.crs!=crs:g=g.to_crs(crs)
    valid=g.geometry.notna()&(~g.geometry.is_empty)&g.geometry.is_valid
    g=g.loc[valid].copy()
    numerical=pd.to_numeric(g.Class_ID,errors="coerce")
    g=g.loc[numerical.notna()].copy()
    g["class_id"]=numerical.loc[g.index].astype(int)
    g["label"]=(g.class_id==1).astype(np.uint8)
    g["area_m2"]=g.geometry.area
    g=g.loc[(g.area_m2>=MIN_AREA_M2)&g.geometry.intersects(bounds)].copy()
    g["crown_id"]=site+":"+g.index.astype(str)
    reg=pd.read_csv(REF/f"01_{site}_split_register.csv",dtype={"crown_id":"string"})
    assert reg.crown_id.is_unique
    assert set(g.crown_id)==set(reg.crown_id),f"{site} annotation differs from locked Notebook 01 split"
    g=g.merge(reg[["crown_id","split","spatial_block"]],on="crown_id",how="left",validate="one_to_one")
    assert g.split.isin(["training","validation","buffered"]).all()
    ANN[site]=g
    SOURCE_FINGERPRINT[site]=dict(image=sha_file(rp),annotation=sha_file(ap),
                                   split=sha_file(REF/f"01_{site}_split_register.csv"))
    print(site,":",len(g),"crowns; train",sum(g.split=="training"),
          "val",sum(g.split=="validation"),"buffered",sum(g.split=="buffered"))
print("Loaded frozen four-site annotations. Pilot and LOSO evaluation are NOT executed.")

def centered_window(g,patch,src):
    cx,cy=g.centroid.x,g.centroid.y
    row,col=rasterio.transform.rowcol(src.transform,cx,cy)
    return Window(int(col)-patch//2,int(row)-patch//2,patch,patch)

def generate_cache03():
    xx=[];yy=[];ss=[];sites=[];ids=[]
    # A crop is selected using a crown centroid, but loss includes only the anchor
    # crown's own pixels. This avoids assigning a neighboring split's labels.
    for site in training_sites:
        with rasterio.open(GEO/SITES[site][0]) as src:
            for split_code,split in ((0,"training"),(1,"validation")):
                g=pilot_indices[site,split]
                for _,rec in tqdm(g.iterrows(),total=len(g),desc=f"{site}/{split}",leave=False):
                    w=centered_window(rec.geometry,PATCH,src)
                    rgb=src.read([1,2,3],window=w,boundless=True,fill_value=0)
                    arr=np.moveaxis(rgb,0,-1).astype(np.uint8)
                    original=arr
                    valid=~((original==0).all(axis=2)|
                            (original==255).all(axis=2)|
                            ((original[:,:,0]==0)&(original[:,:,1]==0)&(original[:,:,2]==255)))
                    mask=rasterize([(rec.geometry,int(rec.label))],out_shape=(PATCH,PATCH),
                                   transform=src.window_transform(w),fill=IGNORE,dtype="uint8")
                    mask[~valid]=IGNORE
                    if int((mask!=IGNORE).sum())<12:continue
                    xx.append(arr);yy.append(mask);ss.append(split_code)
                    sites.append(site);ids.append(str(rec.crown_id))
    assert xx and len(set(ss))==2
    X=np.stack(xx);Y=np.stack(yy);S=np.array(ss,np.uint8)
    SI=np.array(sites);CI=np.array(ids)
    print("Caching",len(X),"patches:",round(X.nbytes/1e6,1),"MB raw RGB",flush=True)
    np.savez_compressed(NPZ,X=X,Y=Y,S=S,SI=SI,CI=CI)
    return X,Y,S,SI,CI

MEAN=np.array([.485,.456,.406],dtype=np.float32)[None,None,:]
STD=np.array([.229,.224,.225],dtype=np.float32)[None,None,:]
def shadow04(x,rng):
    # Spatially broad and smooth illumination change on the crop.
    # Default local mask variation is differentiable and multiplicative, not recoloring.
    h,w=x.shape[:2]
    yy,xx=np.mgrid[:h,:w].astype(np.float32)
    theta=rng.uniform(0,np.pi)
    normal=(np.cos(theta)*(xx-w/2)+np.sin(theta)*(yy-h/2))
    offset=rng.uniform(-.35,.35)*max(h,w)
    edge_width=rng.uniform(20,90)
    transition=np.clip((normal-offset)/edge_width,-12,12)
    weights=1/(1+np.exp(-transition))
    darkness=rng.uniform(.45,.90)
    return np.clip(x*(darkness+(1-darkness)*weights[:,:,None]),0,1)
class PilotDataset04(Dataset):
    def __init__(self,which,variant,epoch=0):
        self.indices=np.flatnonzero(S==which)
        self.variant=variant;self.epoch=epoch;self.training=(which==0)
    def __len__(self):return len(self.indices)
    def __getitem__(self,j):
        index=int(self.indices[j])
        img=X[index].astype(np.float32)/255.
        mask=Y[index].copy()
        site=str(SI[index])
        if self.variant=="M2_sitegain_photometric":
            img=np.clip(img*GAIN[site],0,1)
        if self.training:
            rng=np.random.default_rng(SEED+self.epoch*1_000_003+index*101)
            if rng.random()<.5:img=img[:,::-1,:].copy();mask=mask[:,::-1].copy()
            if rng.random()<.5:img=img[::-1,:,:].copy();mask=mask[::-1,:].copy()
            k=int(rng.integers(0,4))
            img=np.rot90(img,k).copy();mask=np.rot90(mask,k).copy()
            if self.variant=="M0_native_historical":
                img=np.clip(img*rng.uniform(.90,1.10),0,1)
            else:
                img=np.clip(img*rng.uniform(.70,1.30),0,1)
                img=np.clip((img-.5)*rng.uniform(.80,1.20)+.5,0,1)
                img=np.power(img,float(rng.uniform(.75,1.35)))
                img=np.clip(img*rng.uniform(.92,1.08,size=(1,1,3)),0,1)
                if rng.random()<.55:img=shadow04(img,rng)
        valid=mask!=IGNORE
        if valid.sum()==0:raise RuntimeError("Empty training patch after source QA")
        tensor=np.moveaxis(((img-MEAN)/STD).astype(np.float32),-1,0).copy()
        return torch.from_numpy(tensor),torch.from_numpy(mask.astype(np.int64))
def new_model04():
    return smp.Unet(encoder_name="resnet18",encoder_weights="imagenet",
                    in_channels=3,classes=1).to(DEVICE)
def make_loader04(code,variant,epoch):
    ds=PilotDataset04(code,variant,epoch)
    return DataLoader(ds,batch_size=BATCH_SIZE,shuffle=(code==0),
        pin_memory=DEVICE=="cuda",num_workers=NUM_WORKERS)
print("ResNet18 U-Net and photometric augmentation functions initialized.")
print("Final M1 functions initialized; four-site training inventory is configured in Section 13.")

def score_valid04(model,loader):
    model.eval();true=[];pred=[]
    with torch.inference_mode():
        for x,y in loader:
            x=x.to(DEVICE,non_blocking=True)
            with torch.amp.autocast("cuda",enabled=DEVICE=="cuda"):
                out=torch.sigmoid(model(x).squeeze(1))
            mask=(y!=IGNORE)
            true.append(y[mask].numpy().astype(np.uint8))
            pred.append(out.detach().cpu()[mask].numpy().astype(np.float32))
    target=np.concatenate(true);prob=np.concatenate(pred)
    return dict(ap=float(average_precision_score(target,prob)) if target.sum() else np.nan,
        auc=float(roc_auc_score(target,prob)) if len(np.unique(target))==2 else np.nan,
        pixels=int(len(target)))
def train_variant04(variant):
    folder=RUN_OUT/variant;folder.mkdir(parents=True,exist_ok=True)
    progress=folder/"last_training_state.pt";bestfile=folder/"best_model.pt"
    history_file=folder/"epoch_history.csv";marker=folder/"pilot_complete.json"
    if marker.exists() and bestfile.exists() and not FORCE_RESTART:
        print("REUSING COMPLETED PILOT:",variant,flush=True)
        return json.loads(marker.read_text())
    seed_all(SEED)
    model=new_model04()
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
    scaler=torch.amp.GradScaler("cuda",enabled=DEVICE=="cuda")
    history=[];start=1;best=-1.;wait=0
    if progress.exists() and not FORCE_RESTART:
        state=torch.load(progress,map_location=DEVICE,weights_only=False)
        assert state["variant"]==variant and state["run_id"]==RUN_ID
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])
        best=float(state["best"]);wait=int(state["wait"])
        start=int(state["epoch"])+1
        history=state["history"]
        print("RESUMING:",variant,"epoch",start,"best AP",round(best,4),flush=True)
    for epoch in range(start,MAX_EPOCHS+1):
        begin=time.monotonic();model.train();losses=[]
        loader=make_loader04(0,variant,epoch)
        for bi,(x,y) in enumerate(loader,1):
            x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda",enabled=DEVICE=="cuda"):
                logits=model(x).squeeze(1)
                valid=(y!=IGNORE)
                loss=nn.functional.binary_cross_entropy_with_logits(logits[valid],y[valid].float())
            scaler.scale(loss).backward()
            scaler.step(optimizer);scaler.update()
            losses.append(float(loss.item()))
            if bi==1 or bi%25==0 or bi==len(loader):
                print(f"  {variant} epoch {epoch} batch {bi}/{len(loader)} loss {np.mean(losses):.4f}",
                      flush=True)
        validation=score_valid04(model,make_loader04(1,variant,0))
        improved=np.isfinite(validation["ap"]) and validation["ap"]>best
        if improved:
            best=validation["ap"];wait=0
            torch.save(dict(model=model.state_dict(),epoch=epoch,variant=variant,
                            run_id=RUN_ID,best_validation_ap=best),bestfile)
        else:wait+=1
        info=dict(epoch=epoch,train_loss=float(np.mean(losses)),validation_ap=validation["ap"],
                  validation_auc=validation["auc"],validation_pixels=validation["pixels"],
                  seconds=float(time.monotonic()-begin),best_validation_ap=best,wait=wait)
        history.append(info)
        pd.DataFrame(history).to_csv(history_file,index=False)
        torch.save(dict(variant=variant,run_id=RUN_ID,epoch=epoch,
            model=model.state_dict(),optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),
            best=best,wait=wait,history=history),progress)
        print(f"PROGRESS {variant} epoch {epoch}/{MAX_EPOCHS}: loss={info['train_loss']:.4f}, "
              f"AP={info['validation_ap']:.4f}, AUC={info['validation_auc']:.4f}, "
              f"best_AP={best:.4f}, {info['seconds']:.1f}s",flush=True)
        if wait>=PATIENCE:
            print("EARLY STOP:",variant,"after",epoch,"epochs",flush=True);break
    if not bestfile.exists():raise RuntimeError(f"Best checkpoint never saved for {variant}")
    summary=dict(variant=variant,run_id=RUN_ID,heldout=HELDOUT,
                 epochs_completed=len(history),best_validation_ap=best,
                 selection_basis="spatial validation pixels on configured training forests",
                 heldout_labels_used_for_selection=False)
    marker.write_text(json.dumps(summary,indent=2))
    del model,optimizer;gc.collect()
    if DEVICE=="cuda":torch.cuda.empty_cache()
    return summary

FULL_BATCH_SIZE=12

# --------------------- FINAL-MODEL CONFIGURATION ---------------------
FINAL04=OUT/"final_four_site_M1";FINAL04.mkdir(parents=True,exist_ok=True)
FINAL_VARIANT04="M1_native_photometric"
FINAL_MAX_EPOCHS04=20
FINAL_PATIENCE04=5
RUN_FINAL_M1_04=True
RUN_LUKI_04=True
LUKI_RGB04=GEO/"Luki.tif"
LUKI_META04=REF/"01_luki_current_reference_version.json"
assert LUKI_META04.exists(),"First run updated Notebook 01 with the new Luki annotations"
LUKI_MANIFEST04=json.loads(LUKI_META04.read_text())
LUKI_VERSION04=LUKI_MANIFEST04["reference_id"]
LUKI_REFERENCE04=Path(LUKI_MANIFEST04["frozen_copy"])
assert LUKI_REFERENCE04.exists(),"Missing frozen external Luki version"
LUKI_CROWNS04=gpd.read_file(LUKI_REFERENCE04,layer="external_evaluation")
assert len(LUKI_CROWNS04)==int(LUKI_MANIFEST04["crowns"])
assert int(LUKI_CROWNS04.label.sum())==int(LUKI_MANIFEST04["musanga"])
assert LUKI_CROWNS04.crown_id.is_unique
assert LUKI_CROWNS04.crs is not None
with rasterio.open(LUKI_RGB04) as src:
    assert LUKI_CROWNS04.crs==src.crs
print("Final M1 source forests",list(SITES))
print("Luki independent source",LUKI_VERSION04,
      len(LUKI_CROWNS04),"crowns (reserved until model fully frozen)")
print("Final 4-site M1 directory:",FINAL04)


## 13 — Final M1 training on all four original forests (resumable)

Run directly after Section 12 in a fresh GPU runtime. Read or build one four-forest native-RGB crown-centred cache using the frozen Notebook 01 training/validation splits. Train **M1 only** with the previously validated photometric recipe; save epoch checkpoints, select the model using development-forest validation pixel AP, and never consult Luki labels. No preceding pilot or LOSO cells are required.

In [ ]:
FINAL_CONFIG04=dict(stage="four_site_M1_v1",seed=SEED,patch=PATCH,
  training_sites=list(SITES),train_crown_ids={
      site:sorted(ANN[site].loc[ANN[site].split=="training","crown_id"].astype(str).tolist())
      for site in SITES},
  val_crown_ids={
      site:sorted(ANN[site].loc[ANN[site].split=="validation","crown_id"].astype(str).tolist())
      for site in SITES},
  source_fingerprints=SOURCE_FINGERPRINT,
  photo_augmentation="M1_native_photometric_same_as_LOSO",
  max_epochs=FINAL_MAX_EPOCHS04,patience=FINAL_PATIENCE04)
FINAL_RUN_ID04=hashlib.sha256(json.dumps(FINAL_CONFIG04,sort_keys=True).encode()).hexdigest()[:12]
FINAL_RUN04=FINAL04/FINAL_RUN_ID04
FINAL_RUN04.mkdir(parents=True,exist_ok=True)
FINAL_CACHE04=CACHE/f"final_four_site_M1_{FINAL_RUN_ID04}.npz"

def final_prepare04():
    global HELDOUT,training_sites,reference,GAIN,RUN_ID,RUN_OUT,NPZ
    global pilot_indices,TEST_CROWNS,X,Y,S,SI,CI
    global RUN_VARIANTS,MAX_EPOCHS,PATIENCE,BATCH_SIZE,FORCE_RESTART
    # No fifth held-out development site: change training_sites to ALL four.
    HELDOUT="FINAL_FOUR_SITE_M1"
    training_sites=list(SITES)
    reference=np.nan # M1 uses native RGB: no site reference or gain is required
    GAIN={s:1.0 for s in SITES} # M1 never uses site gains
    pilot_indices={(site,sp):ANN[site][ANN[site].split==sp].copy()
                   for site in training_sites for sp in ("training","validation")}
    TEST_CROWNS=gpd.GeoDataFrame()
    RUN_ID=FINAL_RUN_ID04
    RUN_OUT=FINAL_RUN04
    NPZ=FINAL_CACHE04
    RUN_VARIANTS=[FINAL_VARIANT04]
    MAX_EPOCHS=FINAL_MAX_EPOCHS04
    PATIENCE=FINAL_PATIENCE04
    BATCH_SIZE=FULL_BATCH_SIZE
    FORCE_RESTART=False
    if NPZ.exists():
        with np.load(NPZ,allow_pickle=False) as cache:
            X,Y,S,SI,CI=[cache[k] for k in ["X","Y","S","SI","CI"]]
        print("REUSED full four-site training cache:",len(X),flush=True)
    else:
        X,Y,S,SI,CI=generate_cache03()
    assert set(SI)==set(SITES)
    assert (S==0).any() and (S==1).any()
    print("Four-site data:",len(X),"patches",
          int((S==0).sum()),"training",int((S==1).sum()),"validation",flush=True)
    print("Run ID:",RUN_ID,flush=True)

FINAL_CKPT04=FINAL_RUN04/FINAL_VARIANT04/"best_model.pt"
if RUN_FINAL_M1_04:
    assert DEVICE=="cuda","Select a Colab GPU runtime."
    final_prepare04()
    FINAL_TRAIN04=train_variant04(FINAL_VARIANT04)
    assert FINAL_CKPT04.exists()
    (FINAL_RUN04/"04_final_model_contract.json").write_text(json.dumps(dict(
        run_id=FINAL_RUN_ID04,architecture="smp.Unet(resnet18,ImageNet encoder)",
        model=FINAL_VARIANT04,source_forests=list(SITES),
        best_checkpoint=str(FINAL_CKPT04),
        spatial_training_and_validation="Notebook 01 250m blocks/30m buffer",
        class_labels="Class_ID=1 Musanga, other numeric Class_ID non-Musanga",
        best_validation_ap=FINAL_TRAIN04["best_validation_ap"],
        epochs_completed=FINAL_TRAIN04["epochs_completed"],
        sampling="256px native GSD crown-centred single annotated crown pixel loss",
        validation="spatial validation pixels across all four forests",
        luki="Not seen during training, validation, checkpoint selection or hyperparameter tuning",
        source_fingerprints=SOURCE_FINGERPRINT,
        limitation="Not wall-to-wall probability map; spatial seams and shadow patches require further QA"
    ),indent=2))
    print("FINAL FOUR-SITE M1 READY:",FINAL_CKPT04,flush=True)

else:
    print("Training disabled; expecting an existing final checkpoint:",FINAL_CKPT04)


## 14 — Independent external Luki crown ROC-AUC/AP, plus illumination diagnostics

Evaluate **exactly the frozen reference version** printed in Section 12. For each manually delineated crown read a native RGB window centred on the crown, predict with the frozen M1 model, and average probability over its rasterized geometry. Reject crowns whose polygon cannot fit inside the inference window or lack sufficient valid RGB support rather than silently biasing crown statistics. **ROC-AUC and AP remain primary**; F1 at 0.5 is secondary. Also join the actual Luki native-RGB luminance from Notebook 01, and summarize errors by **common 0–1 absolute brightness bins** and **Luki-relative quartiles**.

The crown diagnostic does **not** test broad unannotated shadow areas, ortho seams or model behaviour on full probability maps. Later we will generate those maps and test their suitability for satellite targets.

In [ ]:
def predict_luki_crowns04(model,gdf):
    rows=[]
    model.eval()
    with rasterio.open(LUKI_RGB04) as src:
        for _,rec in tqdm(gdf.iterrows(),total=len(gdf),desc="Final M1 external Luki"):
            win=centered_window(rec.geometry,PATCH,src)
            rgb=src.read([1,2,3],window=win,boundless=True,fill_value=0)
            raw=np.moveaxis(rgb,0,-1).astype(np.uint8)
            allowed=~((raw==0).all(axis=2)|(raw==255).all(axis=2)|
                      ((raw[:,:,0]==0)&(raw[:,:,1]==0)&(raw[:,:,2]==255)))
            inside=geometry_mask([rec.geometry],out_shape=(PATCH,PATCH),
                transform=src.window_transform(win),invert=True)
            # Reject truncated crowns so the inference subset matches reference geometry.
            expected=max(1.,float(rec.geometry.area)/abs(src.transform.a*src.transform.e))
            coverage=float(inside.sum()/expected)
            mask=inside&allowed
            base=dict(site="Luki",crown_id=str(rec.crown_id),label=int(rec.label),
                valid_pixels=int(mask.sum()),estimated_geometry_coverage=coverage)
            if coverage<.90 or mask.sum()<10:
                rows.append(dict(**base,score=np.nan,qa="truncated_or_low_support"))
                continue
            img=raw.astype(np.float32)/255.
            arr=np.moveaxis(((img-MEAN)/STD).astype(np.float32),-1,0).copy()
            with torch.inference_mode(),torch.amp.autocast("cuda",enabled=DEVICE=="cuda"):
                p=torch.sigmoid(model(torch.from_numpy(arr[None]).to(DEVICE))).squeeze().float().cpu().numpy()
            rows.append(dict(**base,score=float(np.mean(p[mask])),qa="ok"))
    return pd.DataFrame(rows)

LUKI_EVAL_DIR04=FINAL_RUN04/"luki_external"/LUKI_VERSION04
LUKI_EVAL_DIR04.mkdir(parents=True,exist_ok=True)
RESULT_PATH04=LUKI_EVAL_DIR04/"04_final_M1_luki_crown_predictions.csv"
if RUN_LUKI_04:
    assert FINAL_CKPT04.exists(),"Run Section 13 to train the 4-site final model first."
    if RESULT_PATH04.exists():
        preds=pd.read_csv(RESULT_PATH04,dtype={"crown_id":"string"})
        print("REUSED version-locked external predictions",RESULT_PATH04)
    else:
        model=new_model04()
        state=torch.load(FINAL_CKPT04,map_location=DEVICE,weights_only=False)
        assert state["run_id"]==FINAL_RUN_ID04
        model.load_state_dict(state["model"])
        preds=predict_luki_crowns04(model,LUKI_CROWNS04)
        preds.to_csv(RESULT_PATH04,index=False)
        del model
        if DEVICE=="cuda":torch.cuda.empty_cache()
    assert len(preds)==len(LUKI_CROWNS04)
    assert preds.crown_id.is_unique
    light=pd.read_csv(REF/"01_crown_illumination.csv",dtype={"crown_id":"string"})
    luki_light=light.loc[light.site=="Luki",["crown_id","median_luminance"]]
    assert luki_light.crown_id.is_unique
    preds=preds.merge(luki_light,on="crown_id",how="left",validate="one_to_one")
    assert preds.median_luminance.notna().all()
    preds["fixed_RGB_bin"]=pd.cut(preds.median_luminance,bins=np.linspace(0,1,11),include_lowest=True)
    quartiles=preds.median_luminance.quantile([.25,.75])
    preds["relative_light"]=np.where(preds.median_luminance<quartiles.loc[.25],
        "darkest_25pct",np.where(preds.median_luminance>quartiles.loc[.75],
        "brightest_25pct","middle_50pct"))
    evaluated=preds[preds.qa=="ok"].copy()
    print("Luki reference crowns",len(preds),"with valid centre patch",len(evaluated),
          "excluded due to truncation/coverage",len(preds)-len(evaluated),flush=True)
    if evaluated.label.nunique()!=2:raise RuntimeError("Both species classes required for AUC/AP")
    p=evaluated.score.to_numpy(float);y=evaluated.label.to_numpy(int)
    q=(p>=.5).astype(int)
    metrics=dict(model="final_four_site_M1",reference_version=LUKI_VERSION04,
        reference_crowns=len(preds),evaluated_crowns=len(evaluated),
        roc_auc=float(roc_auc_score(y,p)),average_precision=float(average_precision_score(y,p)),
        precision_0p5=float(precision_score(y,q,zero_division=0)),
        recall_0p5=float(recall_score(y,q,zero_division=0)),
        f1_0p5=float(f1_score(y,q,zero_division=0)),
        balanced_accuracy_0p5=float(balanced_accuracy_score(y,q)),
        false_positives=int(((y==0)&(q==1)).sum()),
        false_negatives=int(((y==1)&(q==0)).sum()))
    pd.DataFrame([metrics]).to_csv(LUKI_EVAL_DIR04/"04_final_M1_luki_metrics.csv",index=False)
    display(pd.DataFrame([metrics]).round(4))
    details=[]
    for name,col in [("absolute_RGB_bin","fixed_RGB_bin"),("relative_brightness","relative_light")]:
        for value,gg in evaluated.groupby(col,observed=True):
            pos=gg[gg.label==1];neg=gg[gg.label==0]
            details.append(dict(stratum_type=name,stratum=str(value),n=len(gg),
                musanga=len(pos),other=len(neg),
                mean_probability=float(gg.score.mean()),
                musanga_recall=float((pos.score>=.5).mean()) if len(pos) else np.nan,
                other_false_positive_rate=float((neg.score>=.5).mean()) if len(neg) else np.nan,
                auc=float(roc_auc_score(gg.label,gg.score)) if gg.label.nunique()==2 else np.nan,
                ap=float(average_precision_score(gg.label,gg.score)) if gg.label.nunique()==2 else np.nan))
    strata=pd.DataFrame(details)
    strata.to_csv(LUKI_EVAL_DIR04/"04_final_M1_luki_brightness_metrics.csv",index=False)
    display(strata.round(4))
    preds.to_csv(LUKI_EVAL_DIR04/"04_final_M1_luki_crown_predictions_with_brightness.csv",index=False)
    print("NO wall-to-wall probability map has been generated here.",flush=True)
    print("NEXT: independently compare spatial shadow artefacts and satellite target transfer.",flush=True)
else:
    print("Luki evaluation disabled.")
